<a href="https://colab.research.google.com/github/LuisFelipeVelasco/ML_And_Generative_AI_Experiments/blob/main/Notebooks/binary_decision_tree_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**BINARY DECISION TREE ALGORITHM FROM SCRATCH**

**IMPORT LIBRARIES**

In [123]:
import numpy as np
import pandas as pd
from google.colab import sheets


**DEFINE CLASS**

In [133]:
class BinaryDecisionTree:

    def __init__(self):
        """
        Initializes the BinaryDecisionTree instance.

        Attributes:
            tree (list): Stores the tree as a list of node dicts built during fitting.
        """
        self.tree = []

    def get_filter_labels(self, attribute, labels, valid, rule, condition):
        """
        Filters labels based on a threshold comparison applied to an attribute array.

        Parameters:
            attribute (numpy.ndarray): 1-D array of attribute values.
            labels (numpy.ndarray): 1-D array of class labels matching `attribute`.
            valid (int): Branch selector (1 = primary direction, 0 = complementary).
            rule (float): Threshold value for the comparison.
            condition (int): Comparison direction (0 = primary uses `<`, 1 = primary uses `>`).

        Returns:
            numpy.ndarray: Subset of `labels` whose attribute values satisfy the comparison.
        """
        if valid == 1:
            if condition == 0:
                mask_valid_labels = attribute <= rule
            else:
                mask_valid_labels = attribute >= rule
        else:
            if condition == 0:
                mask_valid_labels = attribute >= rule
            else:
                mask_valid_labels = attribute <= rule
        return labels[mask_valid_labels]

    def get_filter_attributes(self, attributes, attribute, valid, rule, condition):
        """
        Filters rows of the attributes DataFrame by a threshold comparison, then drops
        the split column.

        Parameters:
            attributes (pandas.DataFrame): Feature matrix at the current tree level.
            attribute (str): Column name used as the split criterion.
            valid (int): Branch selector (1 = primary direction, 0 = complementary).
            rule (float): Threshold value for the row filter.
            condition (int): Comparison direction (0 = primary uses `<`, 1 = primary uses `>`).

        Returns:
            pandas.DataFrame: Filtered subset of `attributes` with the `attribute` column removed.
        """

        if valid == 1:
            if condition == 0:
                attributes = attributes[attributes[attribute] <= rule]
            else:
                attributes = attributes[attributes[attribute] >= rule]
        else:
            if condition == 0:
                attributes = attributes[attributes[attribute] >= rule]
            else:
                attributes = attributes[attributes[attribute] <= rule]

        return attributes.drop(attribute, axis=1)

    def get_gini_impurity(self, labels):
        """
        Calculates the Gini impurity for a set of class labels.

        Parameters:
            labels (numpy.ndarray): 1-D array of class labels.

        Returns:
            float: Gini impurity in [0, (K-1)/K]. Returns 0 if `labels` is empty.
        """
        if len(labels) == 0:
            return 0
        counts_unique_labels = np.unique(labels, return_counts=True)[1]
        total_labels = sum(counts_unique_labels)
        gini_impurity = sum(
            (counts_unique_labels / total_labels)
            * (1 - (counts_unique_labels / total_labels))
        )
        return gini_impurity

    def get_most_repeated_value(self, labels):
        """
        Returns the majority class label (mode) in the array.

        Parameters:
            labels (numpy.ndarray): 1-D array of class labels.

        Returns:
            scalar: The label with the highest frequency.
        """
        labels_unique, labels_count = np.unique(labels, return_counts=True)
        return labels_unique[np.argmax(labels_count)]

    def get_less_repeated_value(self, labels):
        """
        Returns the minority class label in the array.

        Parameters:
            labels (numpy.ndarray): 1-D array of class labels.

        Returns:
            scalar: The label with the lowest frequency.
        """
        labels_unique, labels_count = np.unique(labels, return_counts=True)
        return labels_unique[np.argmin(labels_count)]

    def get_metrics_attribute(self, attribute, labels):
        """
        Finds the optimal binary split threshold for a single attribute by minimising
        Gini impurity over all candidate midpoints between consecutive sorted values.

        Parameters:
            attribute (numpy.ndarray): 1-D array of attribute values.
            labels (numpy.ndarray): 1-D array of class labels in the same order as `attribute`.

        Returns:
            tuple:
                rule_attribute (list): [threshold, condition] for the optimal split.
                gini_impurity (float): Gini impurity of the winning branch.
                old_matching_samples (int): Number of samples in the winning branch.
        """

        sorted_indices = np.argsort(attribute)[::-1]
        attribute_sorted = attribute.iloc[sorted_indices]
        labels_sorted = labels[sorted_indices]
        old_average = 0
        rule_attribute = []
        gini_impurity = 1
        old_matching_samples = 0
        min_samples = int((len(labels)/3))

        for i in range(len(attribute_sorted) - 1):
            average = (attribute_sorted.iloc[i] + attribute_sorted.iloc[i + 1]) / 2
            if average != old_average:
                old_average = average
                labels_satisfy_less_operation = self.get_filter_labels(
                    attribute_sorted, labels_sorted, 1, average, 0
                )
                labels_satisfy_great_operation = self.get_filter_labels(
                    attribute_sorted, labels_sorted, 1, average, 1
                )
                gini_impurity_great_operation = self.get_gini_impurity(
                    labels_satisfy_great_operation
                )
                gini_impurity_less_operation = self.get_gini_impurity(
                    labels_satisfy_less_operation
                )
                number_matching_great_sample = len(labels_satisfy_great_operation)
                number_matching_less_sample = len(labels_satisfy_less_operation)

                if (
                    gini_impurity_great_operation < gini_impurity_less_operation
                    and gini_impurity_great_operation <= gini_impurity
                    and (
                        number_matching_great_sample >= old_matching_samples
                        or number_matching_great_sample >= min_samples
                    )
                ):
                    gini_impurity = gini_impurity_great_operation
                    rule_attribute = [average, 1]
                    old_matching_samples = number_matching_great_sample

                elif (
                    gini_impurity_less_operation < gini_impurity_great_operation
                    and gini_impurity_less_operation <= gini_impurity
                    and (
                        number_matching_less_sample >= old_matching_samples
                        and number_matching_less_sample >= min_samples
                    )
                ):
                    gini_impurity = gini_impurity_less_operation
                    rule_attribute = [average, 0]
                    old_matching_samples = number_matching_less_sample

                elif (
                    gini_impurity_great_operation == gini_impurity_less_operation
                    and number_matching_less_sample > number_matching_great_sample
                    and gini_impurity_less_operation <= gini_impurity
                    and (
                        number_matching_less_sample >= old_matching_samples
                        or number_matching_less_sample >= min_samples
                    )
                ):
                    gini_impurity = gini_impurity_less_operation
                    rule_attribute = [average, 0]
                    old_matching_samples = number_matching_less_sample

                elif (
                    gini_impurity_great_operation == gini_impurity_less_operation
                    and number_matching_great_sample > number_matching_less_sample
                    and gini_impurity_great_operation <= gini_impurity
                    and (
                        number_matching_great_sample >= old_matching_samples
                        or number_matching_great_sample >= min_samples
                    )
                ):
                    gini_impurity = gini_impurity_great_operation
                    rule_attribute = [average, 1]
                    old_matching_samples = number_matching_great_sample

        return rule_attribute, gini_impurity, old_matching_samples

    def get_properties_best_attribute(self, attributes, labels):
        """
        Selects the best feature and its optimal split rule from the current attribute set.

        Parameters:
            attributes (pandas.DataFrame): Feature matrix at the current tree level.
            labels (numpy.ndarray): Class labels for the current sample subset.

        Returns:
            tuple:
                best_attribute (str or int): Column name of the selected feature.
                rule_best_attribute (list): [threshold, condition] for the optimal split.
        """
        best_attribute = 0
        rule_best_attribute = []
        gini_impurity_best_attribute = 1
        great_matching_samples = 0
        min_samples = int((len(labels) * 25) / 100)

        for i in range(0, len(attributes.columns)):
            attribute = attributes.iloc[:, i]
            rule_attribute, gini_impurity_attribute, matching_samples = (
                self.get_metrics_attribute(attribute, labels)
            )
            # print(rule_attribute)
            # print(gini_impurity_attribute)
            # print(matching_samples)
            if gini_impurity_attribute <= gini_impurity_best_attribute and (
                matching_samples > great_matching_samples
                or great_matching_samples >= min_samples
            ):
                gini_impurity_best_attribute = gini_impurity_attribute
                rule_best_attribute = rule_attribute
                great_matching_samples = matching_samples
                best_attribute = attributes.columns[i]

        return best_attribute, rule_best_attribute

    def fit(self, attributes, labels):
        """
        Trains the decision tree on the provided dataset.

        Parameters:
            attributes (pandas.DataFrame): Training feature matrix.
            labels (numpy.ndarray): 1-D array of class labels for each training sample.

        Returns:
            list: The fully constructed tree as a list of node dicts.
        """
        root_attribute, rule = self.get_properties_best_attribute(attributes, labels)
        root = self.get_node("root", -1, 0, root_attribute, rule[0], rule[1])
        self.tree.append(root)
        self.generate_nodes(root, attributes, labels)
        return self.tree

    def generate_nodes(self, last_node, attributes, labels):
        """
        Recursively generates and appends child nodes for a given parent decision node.

        Parameters:
            last_node (dict): Parent node dictionary containing the split attribute and rule.
            attributes (pandas.DataFrame): Feature matrix available at this depth.
            labels (numpy.ndarray): Class labels for the samples reaching this node.

        Returns:
            None
        """
        attribute = last_node["value"][0]
        rule = last_node["value"][1][0]
        condition = last_node["value"][1][1]
        if len(attributes.columns) > 1:
            for i in range(0, 2):
                labels_attribute = self.get_filter_labels(attributes[attribute], labels, i, rule, condition)
                new_attributes = self.get_filter_attributes(attributes, attribute, i, rule, condition)
                print("rule: ", rule)
                print("condition: ", condition)
                print("valid: " , i)
                print(new_attributes)
                if len(labels_attribute) == 0:
                    label = self.get_most_repeated_value(labels)
                    new_node = self.get_node("conclusion", attribute, i, 0, label, 0)
                    self.tree.append(new_node)

                else:
                  gini_impurity = self.get_gini_impurity(labels_attribute)
                  if gini_impurity == 0:
                      label = self.get_most_repeated_value(labels_attribute)
                      new_node = self.get_node("conclusion", attribute, i, 0, label, 0)
                      self.tree.append(new_node)
                  else:
                      new_best_attribute, rule_best_attribute = (
                          self.get_properties_best_attribute(
                              new_attributes, labels_attribute
                          )
                      )
                      new_node = self.get_node(
                          "decision",
                          attribute,
                          i,
                          new_best_attribute,
                          rule_best_attribute[0],
                          rule_best_attribute[1],
                      )
                      self.tree.append(new_node)
                      self.generate_nodes(new_node, new_attributes, labels_attribute)
        else:
            labels_attribute = self.get_filter_labels(
                attributes[attribute], labels, 1, rule, condition
            )
            print("HOLAAAAA")
            print(labels_attribute)
            label = self.get_most_repeated_value(labels_attribute)
            new_node = self.get_node("conclusion", attribute, 1, 0, label, 0)
            self.tree.append(new_node)
            labels_attribute = self.get_filter_labels(
                attributes[attribute], labels, 0, rule, condition
            )
            print(labels_attribute)
            label = self.get_less_repeated_value(labels_attribute)
            new_node = self.get_node("conclusion", attribute, 0, 0, label, 0)
            self.tree.append(new_node)

    def get_node(self, type, parent_attribute, filter, idx, rule, condition):
        """
        Constructs and returns a node dictionary for a single tree element.

        Parameters:
            type (str): Node role — "root", "decision", or "conclusion".
            parent_attribute (str or int): Column name of the parent's split attribute, or -1 for root.
            filter (int): Branch direction from the parent (1 = primary, 0 = complementary).
            idx (str or int): Column name to split on (decision/root), or 0 (conclusion).
            rule (float or scalar): Split threshold (decision/root) or predicted class label (conclusion).
            condition (int): Comparison direction flag (0 = primary uses `<`, 1 = primary uses `>`).

        Returns:
            dict: Node dict with keys "type", "parent_attribute", "filter", and "value".
        """
        return {
            "type": type,
            "parent_attribute": parent_attribute,
            "filter": filter,
            "value": [idx, [rule, condition]],
        }

**CHARGE DATASET**

In [125]:
!wget https://raw.githubusercontent.com/LuisFelipeVelasco/ML_And_Generative_AI_Experiments/main/Sources/iris.csv
df=pd.read_csv('iris.csv')
sheet = sheets.InteractiveSheet(df=df)

--2026-06-30 00:07:12--  https://raw.githubusercontent.com/LuisFelipeVelasco/ML_And_Generative_AI_Experiments/main/Sources/iris.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3858 (3.8K) [text/plain]
Saving to: ‘iris.csv.1’

iris.csv.1          100%[===================>]   3.77K  --.-KB/s    in 0s      

2026-06-30 00:07:12 (42.4 MB/s) - ‘iris.csv.1’ saved [3858/3858]

https://docs.google.com/spreadsheets/d/1MbxPDjHBF47ku1SwP22w8EkQOsgiVoux6pgwcLjxoSo/edit#gid=0


**TEST CLASS**

In [134]:
labels=df["species"].values
attributes=df.iloc[:,:4]
model=BinaryDecisionTree()
tree=model.fit(attributes,labels)
print(tree)

rule:  0.8
condition:  0
valid:  0
     sepal_length  sepal_width  petal_length  petal_width
50            7.0          3.2           4.7          1.4
51            6.4          3.2           4.5          1.5
52            6.9          3.1           4.9          1.5
53            5.5          2.3           4.0          1.3
54            6.5          2.8           4.6          1.5
..            ...          ...           ...          ...
145           6.7          3.0           5.2          2.3
146           6.3          2.5           5.0          1.9
147           6.5          3.0           5.2          2.0
148           6.2          3.4           5.4          2.3
149           5.9          3.0           5.1          1.8

[100 rows x 4 columns]
rule:  1.85
condition:  1
valid:  0
     sepal_length  sepal_width  petal_length  petal_width
50            7.0          3.2           4.7          1.4
51            6.4          3.2           4.5          1.5
52            6.9          3.1     

**TEST GET RULE ATTRIBUTE**

In [47]:
labels=df["species"].values
attributes=df.iloc[:,3]
model=BinaryDecisionTree()
print(labels)
print(attributes)
best_attribute=model.get_metrics_attribute(attributes,labels)
print(best_attribute)

['setosa' 'setosa' 'setosa' 'setosa' 'setosa' 'setosa' 'setosa' 'setosa'
 'setosa' 'setosa' 'setosa' 'setosa' 'setosa' 'setosa' 'setosa' 'setosa'
 'setosa' 'setosa' 'setosa' 'setosa' 'setosa' 'setosa' 'setosa' 'setosa'
 'setosa' 'setosa' 'setosa' 'setosa' 'setosa' 'setosa' 'setosa' 'setosa'
 'setosa' 'setosa' 'setosa' 'setosa' 'setosa' 'setosa' 'setosa' 'setosa'
 'setosa' 'setosa' 'setosa' 'setosa' 'setosa' 'setosa' 'setosa' 'setosa'
 'setosa' 'setosa' 'versicolor' 'versicolor' 'versicolor' 'versicolor'
 'versicolor' 'versicolor' 'versicolor' 'versicolor' 'versicolor'
 'versicolor' 'versicolor' 'versicolor' 'versicolor' 'versicolor'
 'versicolor' 'versicolor' 'versicolor' 'versicolor' 'versicolor'
 'versicolor' 'versicolor' 'versicolor' 'versicolor' 'versicolor'
 'versicolor' 'versicolor' 'versicolor' 'versicolor' 'versicolor'
 'versicolor' 'versicolor' 'versicolor' 'versicolor' 'versicolor'
 'versicolor' 'versicolor' 'versicolor' 'versicolor' 'versicolor'
 'versicolor' 'versicolor' 'v

**TEST GET BEST ATTRIBUTE**

In [81]:
labels=df.iloc[:,4].values
new_attributes=df.iloc[:,:4]
model=BinaryDecisionTree()
best_attribute,best_rule=model.get_properties_best_attribute(new_attributes,labels)
print(best_attribute)
print(best_rule)

131    7.9
135    7.7
118    7.7
117    7.7
122    7.7
      ... 
41     4.5
38     4.4
42     4.4
8      4.4
13     4.3
Name: sepal_length, Length: 150, dtype: float64
[7.9 7.7 7.7 7.7 7.7 7.6 7.4 7.3 7.2 7.2 7.2 7.1 7.  6.9 6.9 6.9 6.9 6.8
 6.8 6.8 6.7 6.7 6.7 6.7 6.7 6.7 6.7 6.7 6.6 6.6 6.5 6.5 6.5 6.5 6.5 6.4
 6.4 6.4 6.4 6.4 6.4 6.4 6.3 6.3 6.3 6.3 6.3 6.3 6.3 6.3 6.3 6.2 6.2 6.2
 6.2 6.1 6.1 6.1 6.1 6.1 6.1 6.  6.  6.  6.  6.  6.  5.9 5.9 5.9 5.8 5.8
 5.8 5.8 5.8 5.8 5.8 5.7 5.7 5.7 5.7 5.7 5.7 5.7 5.7 5.6 5.6 5.6 5.6 5.6
 5.6 5.5 5.5 5.5 5.5 5.5 5.5 5.5 5.4 5.4 5.4 5.4 5.4 5.4 5.3 5.2 5.2 5.2
 5.2 5.1 5.1 5.1 5.1 5.1 5.1 5.1 5.1 5.1 5.  5.  5.  5.  5.  5.  5.  5.
 5.  5.  4.9 4.9 4.9 4.9 4.9 4.9 4.8 4.8 4.8 4.8 4.8 4.7 4.7 4.6 4.6 4.6
 4.6 4.5 4.4 4.4 4.4 4.3]
7.800000000000001
1
7.7
7.7
7.7
7.65
1
7.5
1
7.35
1
7.25
1
7.2
7.2
7.15
1
7.05
1
6.95
6.9
6.9
6.9
6.85
6.8
6.8
6.75
6.7
6.7
6.7
6.7
6.7
6.7
6.7
6.65
6.6
6.55
6.5
6.5
6.5
6.5
6.45
6.4
6.4
6.4
6.4
6.4
6.4
6.35
6.3
6.3
6.3
6.